# 20.17 — LLMOps & prompt management

LLMOps treats prompts, retrieval context, tools, and evals as versioned production artifacts rather than informal strings.

Experiment tracking supplies versioned comparisons, and vector search supplies retrieval context that prompts consume. Evaluation harnesses, online experiments, and cost optimization become the release machinery for prompts. Save a copy to Drive to edit.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

SEED = 17
rng = np.random.default_rng(SEED)
np.set_printoptions(precision=4, suppress=True)

## The concept, built once

Prompt cost and pass rate: $C=t_{in}c_{in}+t_{out}c_{out}$ and $\text{pass}=n_{pass}/n$.

The reusable method versions the prompt package, enforces context budgets, scores fixed outcomes, and computes cost per passed request without calling an external LLM.

In [ ]:
def prompt_eval(prompt_version, context_tokens, outcomes, input_price=0.000002, output_price=0.000006, output_tokens=220, budget=3000):
    token_array = np.asarray(context_tokens, dtype=float)
    outcome_array = np.asarray(outcomes, dtype=float)
    packed_tokens = np.minimum(token_array, budget)
    truncated = token_array > budget
    effective_pass = outcome_array.copy()
    effective_pass[truncated] = effective_pass[truncated] * 0.55
    input_cost = packed_tokens * input_price
    output_cost = np.full_like(packed_tokens, output_tokens * output_price, dtype=float)
    total_cost = input_cost + output_cost
    pass_rate = float(np.mean(effective_pass))
    passed = float(np.sum(effective_pass))
    cost_per_pass = float(np.sum(total_cost) / max(passed, 1))
    return {
        "version": prompt_version,
        "tokens": packed_tokens,
        "truncated": truncated,
        "total_cost": total_cost,
        "pass_rate": pass_rate,
        "cost_per_pass": cost_per_pass,
    }

lesson_tokens = 500 + 1800 + 300
lesson_cost = lesson_tokens * 0.000002
lesson_lift = 0.74 - 0.69
fallback_calls = 10000 * 0.08
regression_rate = 3 / 60
assert lesson_tokens == 2600
assert abs(lesson_cost - 0.0052) < 1e-12
assert abs(lesson_lift - 0.05) < 1e-12
assert fallback_calls == 800
assert abs(regression_rate - 0.05) < 1e-12
print("tokens", lesson_tokens)
print("input cost", lesson_cost)
print("pass-rate lift", lesson_lift)

The same method now becomes the single evaluation API. It returns the operational artifact and the topic metric so the D1 proof scales to D2–D5.

In [ ]:
print("Build-once assertions passed for 20.17")

## The dataset ladder

Family F17 uses an inline D1–D5 systems workload ladder. Each rung increases operational realism while keeping the computation CPU-only, seeded, and NumPy based.

In [ ]:
def make_prompt_cases(n, rng, base_tokens, pass_rate, spike_rate=0.0, malformed_rate=0.0):
    system = rng.normal(base_tokens[0], 40, size=n).clip(100, None)
    retrieval = rng.normal(base_tokens[1], 250, size=n).clip(0, None)
    user = rng.normal(base_tokens[2], 80, size=n).clip(20, None)
    tokens = system + retrieval + user
    spikes = rng.random(n) < spike_rate
    tokens[spikes] = tokens[spikes] + rng.integers(1500, 3500, size=np.sum(spikes))
    malformed = rng.random(n) < malformed_rate
    probability = np.full(n, pass_rate)
    probability[malformed] = probability[malformed] - 0.25
    probability[spikes] = probability[spikes] - 0.10
    probability = np.clip(probability, 0.05, 0.98)
    outcomes = rng.random(n) < probability
    routes = rng.choice(["support", "sales", "policy"], size=n, p=[0.55, 0.30, 0.15])
    return {"tokens": tokens, "outcomes": outcomes.astype(float), "routes": routes, "spikes": spikes}

def build_prompt_ladder(seed=17):
    rng = np.random.default_rng(seed)
    ladder = []
    ladder.append({"name": "D1 hand prompt cases", "cases": {"tokens": np.array([2600, 2400, 2800, 2100, 2700], dtype=float), "outcomes": np.array([1, 1, 0, 1, 1], dtype=float), "routes": np.array(["support", "support", "sales", "policy", "sales"]), "spikes": np.array([False, False, False, False, False])}, "budget": 3000, "load": 5})
    ladder.append({"name": "D2 clean 1k eval requests", "cases": make_prompt_cases(1000, rng, (500, 1800, 300), 0.74), "budget": 3000, "load": 1000})
    ladder.append({"name": "D3 distractors and token spikes", "cases": make_prompt_cases(2500, rng, (520, 2100, 330), 0.72, 0.12, 0.06), "budget": 3000, "load": 2500})
    ladder.append({"name": "D4 route sampled traffic", "cases": make_prompt_cases(6000, rng, (480, 1900, 360), 0.76, 0.08, 0.03), "budget": 3200, "load": 6000})
    ladder.append({"name": "D5 production routing simulation", "cases": make_prompt_cases(12000, rng, (550, 2600, 450), 0.78, 0.22, 0.08), "budget": 3000, "load": 12000})
    return ladder

ladder = build_prompt_ladder()
for rung in ladder:
    cases = rung["cases"]
    print(rung["name"], "n", len(cases["tokens"]), "budget", rung["budget"], "sample tokens", np.round(cases["tokens"][:5], 1).tolist())

## Run the same method across D1–D5

The metric is collected in the same way for every rung, so changes across the ladder reflect workload complexity rather than a different measurement.

In [ ]:
results = []
for rung in ladder:
    cases = rung["cases"]
    result = prompt_eval(rung["name"], cases["tokens"], cases["outcomes"], budget=rung["budget"])
    results.append({"name": rung["name"], "metric": result["cost_per_pass"], "artifact": result, "load": rung["load"], "pass_rate": result["pass_rate"]})

print("rung | cost per passed request | pass rate | load")
for item in results:
    print(item["name"], round(item["metric"], 6), round(item["pass_rate"], 3), item["load"])

## Results visualization

The closing figure has operational panels for each rung plus a metric-vs-load curve.

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(18, 6))
for col, item in enumerate(results):
    artifact = item["artifact"]
    axes[0, col].hist(artifact["tokens"], bins=30, color="tab:green")
    axes[0, col].set_title(item["name"].split()[0] + " tokens")
    axes[0, col].set_xlabel("packed tokens")
    axes[0, col].set_ylabel("requests")
    cumulative_cost = np.cumsum(artifact["total_cost"][:1000])
    axes[1, col].plot(cumulative_cost, color="tab:red")
    axes[1, col].set_xlabel("request")
    axes[1, col].set_ylabel("cost")
fig.suptitle("Operational panels: token load and prompt cost")
plt.tight_layout()

fig, ax = plt.subplots(figsize=(7, 4))
loads = [item["load"] for item in results]
metrics = [item["metric"] for item in results]
ax.plot(loads, metrics, marker="o")
ax.set_xlabel("prompt load")
ax.set_ylabel("cost per passed request")
ax.set_title("Metric vs context load")
plt.show()

## Pitfall on the hardest rung

Reproduce the lesson pitfall on D5, then apply the operational fix and compare the metric.

In [ ]:
d5 = ladder[-1]
cases = d5["cases"]
naive = prompt_eval("naive", cases["tokens"], cases["outcomes"], budget=2200)
fixed_tokens = np.minimum(cases["tokens"], 3000)
fixed = prompt_eval("budget-aware", fixed_tokens, cases["outcomes"], budget=3000)
print("naive truncation rate", round(float(np.mean(naive["truncated"])), 3))
print("naive cost per pass", round(naive["cost_per_pass"], 6))
print("fixed truncation rate", round(float(np.mean(fixed["truncated"])), 3))
print("fixed cost per pass", round(fixed["cost_per_pass"], 6))

## Evaluate it + Practice

- Metric: cost per passed request, with a no-budget baseline that truncates blindly.
- Sanity check: fixed lesson arithmetic gives 2,600 tokens and $0.0052 input cost.
- Ablation: remove budget-aware retrieval packing and watch pass rate drop on token spikes.
- Failure signals: high truncation rate, rising cost per pass, and golden-case regressions.

Practice prompts:
1. Change one workload knob on D3 and predict whether the metric should improve or degrade.
2. Add one slice or route to D4 and explain which operational panel should change.
3. Tighten the D5 budget or threshold and rerun only after saving your own copy.